# Manual RAG Tests
Notebook para probar manualmente health, ingesta MinIO y consulta con cache/RAG.

In [ ]:
import json
import requests

BASE_URL = 'http://127.0.0.1:8000'
API_PREFIX = '/api/v1'

In [2]:
# 1) Health check
resp = requests.get(f'{BASE_URL}{API_PREFIX}/health', timeout=20)
print(resp.status_code)
print(resp.json())

200
{'status': 'ok'}


In [3]:
# 2) Ingesta de documento desde path local (PDF/imagen/texto)
local_pdf_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf"  # <- cambia esto
ingest_payload = {
    'file_path': local_pdf_path,
    # 'minio_path': 'minio://my-bucket/contexto/ley-ejemplo.pdf',  # alternativa
    'presentado_por': 'Dip. Juan Perez',
    'proyecto_tipo': 'modificacion',  # 'base' o 'modificacion'
    'ley_base': 'Ley 27.275',
    'iniciado_en': 'Camara de Diputados',
    'expediente_diputados': '1234-D-2026',
    'expediente_senado': '5678-S-2026',
    'publicado_en': 'Boletin Oficial de la Republica Argentina',
    'fecha': '2026-04-10',
    'ley_numero': '27.804',
    'chunk_size': 700,
    'chunk_overlap': 120,
    'replace_existing': True,
    'metadata': {'fuente': 'manual-notebook', 'tipo': 'normativa'}
}

resp = requests.post(
    f'{BASE_URL}{API_PREFIX}/rag/ingest/minio',
    json=ingest_payload,
    timeout=120
 )
print(resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

200
{
  "document_id": "d060b6f604fbfd7f64af31c013dc1561697f8c78",
  "source_uri": "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf",
  "chunks_inserted": 66,
  "bytes_read": 794785,
  "db_path": "app/data/rag.sqlite3"
}


### Importante sobre file_path y metadatos
El backend valida que el archivo esté dentro de los paths permitidos en `LOCAL_FILE_INGEST_BASE_PATHS` del `.env`.
Si te devuelve 403, agrega la carpeta del PDF en esa variable.

Metadatos legislativos:
- Si los envías en payload, se usan como fuente principal.
- Si faltan, el backend intenta inferirlos automáticamente desde el texto OCR/PDF.
- Recomiendo validar manualmente los campos inferidos (`expediente_*`, `ley_numero`, `fecha`).

In [ ]:
# 3) Consulta al endpoint de chat (modo search)
chat_payload = {
    'prompt': 'Es cierto que la ley de glaciares permite vender los Glaciares?'
}

resp = requests.post(
    f'{BASE_URL}{API_PREFIX}/agent/chat',
    json=chat_payload,
    timeout=120
)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

KeyboardInterrupt: 

In [ ]:
%pip install python-dotenv langchain langchain-openai langchain-community  langchain-openai langchain-core

  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langgraph-1.1.6-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.13-py3-none-any.whl.metadata (1.6 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenli

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
openai_api_key

In [24]:
import getpass

if not openai_api_key:
    openai_api_key = getpass.getpass()

In [ ]:

from langchain_core.prompts import ChatPromptTemplate

UserPrompt = "Es cierto que la ley de glaciares permite vender los Glaciares?"


prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Responde únicamente utilizando el contenido proporcionado como única fuente de verdad. \n\nContexto:\n{contexto}",
        ),
        ("human", "{query}"),
    ]
)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

modelo = ChatOpenAI(model="gpt-4.1-nano", temperature=0.2, api_key=openai_api_key)

In [ ]:
import sqlite3

ingest_resp = requests.post(
    f'{BASE_URL}{API_PREFIX}/rag/ingest/minio',
    json=ingest_payload,
    timeout=120
)
ingest_data = ingest_resp.json()
print(ingest_resp.status_code)
print(json.dumps(ingest_data, indent=2, ensure_ascii=False))

db_path = ingest_data["db_path"]  # viene de la respuesta de ingesta
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute(
    """
SELECT document_id, source_uri, chunk_index, metadata_json
FROM rag_chunks
ORDER BY id DESC
LIMIT 5
"""
)

rows = cur.fetchall()
for r in rows:
    print("document_id:", r[0])
    print("source_uri:", r[1])
    print("chunk_index:", r[2])
    print("metadata:", json.dumps(json.loads(r[3]), indent=2, ensure_ascii=False))
    print("-" * 80)

conn.close()

200
{
  "document_id": "d060b6f604fbfd7f64af31c013dc1561697f8c78",
  "source_uri": "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf",
  "chunks_inserted": 66,
  "bytes_read": 794785,
  "db_path": "app/data/rag.sqlite3"
}
document_id: d060b6f604fbfd7f64af31c013dc1561697f8c78
source_uri: /Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf
chunk_index: 65
metadata: {
  "metadata_schema_version": "rag.v1",
  "document_id": "d060b6f604fbfd7f64af31c013dc1561697f8c78",
  "source_uri": "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf",
  "source_kind": "local",
  "source_name": "104088.pdf",
  "source_file_type": "pdf",
  "source_sha256": "5ae678e41ccb8d638caa0e0d0bc7a7a1b562af337f0d6b5ddbe2bf399a1b2ead",
  "bytes_read": 794785,
  "used_ocr": false,
  "ocr_reason": "PDF text extraction succeeded",
  "chunk_strategy": "article",
  "chunk_count": 66,
  "ingested_at_utc": "2026-04-10T21:39:06.054423+00:00",
  "presentado

## Nota
Si no esta levantada la API, ejecuta en terminal:
`uvicorn app.main:app --reload --host 0.0.0.0 --port 8000`